In [1]:
import pandas as pd
from pathlib import Path

ARTIFACTS_DIR = Path("../artifacts")

ml_table = pd.read_csv(ARTIFACTS_DIR / "ml_table.csv")

print("Shape:", ml_table.shape)
display(ml_table.head())

Shape: (99441, 21)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,customer_state,item_count,total_item_price,total_freight_value,unique_products,unique_sellers,total_payment_value,payment_records,max_installments,payment_types
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,SP,1.0,29.99,8.72,1.0,1.0,38.71,3.0,1.0,"credit_card, voucher"
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,BA,1.0,118.70,22.76,1.0,1.0,141.46,1.0,1.0,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,GO,1.0,159.90,19.22,1.0,1.0,179.12,1.0,3.0,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,RN,1.0,45.00,27.20,1.0,1.0,72.20,1.0,1.0,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,SP,1.0,19.90,8.72,1.0,1.0,28.62,1.0,1.0,credit_card


In [2]:
ml_table[
    [
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].dtypes

order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [3]:
ml_table["order_delivered_customer_date"] = pd.to_datetime(
    ml_table["order_delivered_customer_date"],
    errors="coerce"
)

ml_table["order_estimated_delivery_date"] = pd.to_datetime(
    ml_table["order_estimated_delivery_date"],
    errors="coerce"
)

print(ml_table[
    [
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].dtypes)

order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [4]:
print(
    "Missing actual delivery dates:",
    ml_table["order_delivered_customer_date"].isna().sum()
)

print(
    "Missing estimated delivery dates:",
    ml_table["order_estimated_delivery_date"].isna().sum()
)

Missing actual delivery dates: 2965
Missing estimated delivery dates: 0


In [5]:
missing_actual = ml_table[
    ml_table["order_delivered_customer_date"].isna()
]

print(missing_actual["order_status"].value_counts())

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [6]:
labeled_table = ml_table[
    ml_table["order_delivered_customer_date"].notna()
].copy()

print("Rows eligible for labeling:", len(labeled_table))

Rows eligible for labeling: 96476


In [7]:
labeled_table["label"] = (
    labeled_table["order_delivered_customer_date"]
    > labeled_table["order_estimated_delivery_date"]
).map({
    True: "Late",
    False: "On-time"
})

display(
    labeled_table[
        [
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "label"
        ]
    ].head()
)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,label
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,On-time
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,On-time
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,On-time
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,On-time
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,On-time


In [8]:
print(labeled_table["label"].value_counts())

print(
    labeled_table["label"].value_counts(normalize=True) * 100
)

label
On-time    88649
Late        7827
Name: count, dtype: int64
label
On-time    91.887101
Late        8.112899
Name: proportion, dtype: float64


In [9]:
manual_check = labeled_table[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "label"
    ]
].sample(10, random_state=42)

display(manual_check)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,label
22500,c58cff333993bb6b7161d7ec1350eef3,2018-04-06 02:32:49,2018-04-18,On-time
68941,87673b5ccb20de0a91c28cc461105d76,2018-05-23 15:28:28,2018-05-30,On-time
23988,80b430d0029bb33110ac31d60e87e0b8,2017-12-07 18:43:46,2017-12-27,On-time
31303,580603672a21252f21fa8a8b4ca85986,2018-04-26 17:44:27,2018-05-08,On-time
36131,c09f32e7ba9b4a134455b36eeff8fff3,2018-04-13 17:32:07,2018-04-24,On-time
58330,462240cb1ec4e5db517e73fff57ebfc0,2017-12-14 18:56:14,2017-12-20,On-time
95960,5bc2a7b8f0817443a86b69461e742cc9,2018-01-12 21:59:18,2018-02-01,On-time
56131,4ef8f514f95bb0a41f58965ea04e7027,2017-05-26 15:59:47,2017-06-07,On-time
56750,587904dc1c873ebfb6078160b4819d8e,2017-12-06 19:43:34,2017-12-15,On-time
92212,29c3b79aace1b72a82b1232bf494e16f,2018-04-28 15:51:50,2018-01-24,Late


In [10]:
manual_check = manual_check.copy()

manual_check["delay_days"] = (
    manual_check["order_delivered_customer_date"]
    - manual_check["order_estimated_delivery_date"]
).dt.days

display(manual_check)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,label,delay_days
22500,c58cff333993bb6b7161d7ec1350eef3,2018-04-06 02:32:49,2018-04-18,On-time,-12
68941,87673b5ccb20de0a91c28cc461105d76,2018-05-23 15:28:28,2018-05-30,On-time,-7
23988,80b430d0029bb33110ac31d60e87e0b8,2017-12-07 18:43:46,2017-12-27,On-time,-20
31303,580603672a21252f21fa8a8b4ca85986,2018-04-26 17:44:27,2018-05-08,On-time,-12
36131,c09f32e7ba9b4a134455b36eeff8fff3,2018-04-13 17:32:07,2018-04-24,On-time,-11
58330,462240cb1ec4e5db517e73fff57ebfc0,2017-12-14 18:56:14,2017-12-20,On-time,-6
95960,5bc2a7b8f0817443a86b69461e742cc9,2018-01-12 21:59:18,2018-02-01,On-time,-20
56131,4ef8f514f95bb0a41f58965ea04e7027,2017-05-26 15:59:47,2017-06-07,On-time,-12
56750,587904dc1c873ebfb6078160b4819d8e,2017-12-06 19:43:34,2017-12-15,On-time,-9
92212,29c3b79aace1b72a82b1232bf494e16f,2018-04-28 15:51:50,2018-01-24,Late,94


In [11]:
output_path = ARTIFACTS_DIR / "labeled_table.csv"

labeled_table.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: ..\artifacts\labeled_table.csv
